# 45장 · 질문 전체 일치 검색의 기준선과 빈 결과

검색 기준선은 개선 효과를 비교할 첫 구현입니다. 답변과 출처를 함께 돌려주고, 관련 문서를 놓친 질문도 남깁니다.

고정 질문을 검색·답변·출처 표시까지 연결해 중간 프로젝트의 기준선을 만듭니다. 예제는 로컬 규칙으로 흐름을 재현하며 Chroma 검색 품질과 모델의 답변 품질은 별도로 측정합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요.


## 팀 질문을 기준선 검색기에 넣기

평가표와 함께 받은 팀 원문을 검색 대상으로 삼습니다. 문장이 정확히 일치할 때만 정답 후보를 반환하는 기준선에 같은 질문을 모두 넣어 보세요.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/midterm/chapter44.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter45-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/midterm/chapter44.json"
sample_file = here / "chapter45-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['cases'])

교재 예시 chapter45-start.json
[{'id': 'Q01', 'question': '과제는 어디에 내나요?', 'action': '답변', 'expected': ['p2']}, {'id': 'Q02', 'question': '내 점수는 몇 점인가요?', 'action': '보류', 'expected': []}, {'id': 'Q03', 'question': '과제 업로드가 실패했어요. 어떻게 해야 하나요?', 'action': '답변', 'expected': ['p3']}]


두 번째 줄은 읽은 파일의 `cases` 값입니다.

## retrieve()로 문서 본문·출처 조회

가상 안내 p2가 실린 수업용 페이지판은 1쪽입니다. 문단 ID를 course-guide-v1:p2로 연결하고 본문·출처·쪽수를 함께 넘깁니다. 모의 검색은 Q01의 정확한 문구만 받습니다.


### 셀 01 · retrieve()로 문서 본문·출처 조회

도서관 예제의 library-1 대신 팀 자료의 문서 ID가 보이는지 확인하세요.

기준선 한 행을 만들 때 질문, 기대 문단, 검색이 반환한 원문, 사용자에게 표시할 답을 따로 적으세요. 반환된 원문의 한 구절을 직접 가리킬 수 있는지 확인한 뒤, 화면용으로 짧게 다듬은 문장을 마지막 칸에 씁니다.


In [1]:
submit_doc = {'id': 'course-guide-v1:p2',
 'text': '과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다.',
 'source': '가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html)',
 'page': 1}

def retrieve(question, k=2):
    if question.strip() == '과제는 어디에 내나요?':
        return [submit_doc][:k]
    return []

question = '과제는 어디에 내나요?'
found = retrieve(question)
source = found[0]
print(source['text'])
print(source['id'], source['page'])

과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다.
course-guide-v1:p2 1


출력은 제출 방법 원문, course-guide-v1:p2, 1입니다. 문단 번호 2를 쪽수에 넣으면 학생이 맞는 쪽으로 갈 수 없습니다. 완성 화면에서는 “과제함에 제출하고 파일 이름과 등록 시간을 확인합니다”라는 가상 답 옆에 이 출처를 표시합니다.

둘째 줄에서 course-guide-v1:p2는 문단 ID이고, 그 뒤의 1은 수업용 페이지판에서의 쪽 번호입니다. 다른 버전의 운영 안내에서 과제 제출 위치가 바뀌면 답변 문구만 비교해서는 어느 안내를 사용했는지 추적하기 어렵습니다. 문서 버전과 문단 ID를 실행 기록에 함께 남기는 이유입니다.


#### 기준선 검색에서 문단 하나가 선택되는 조건

```mermaid
flowchart TD
nq["질문"]
  nequal["전체 비교"]
  nfound["문서 반환"]
  nempty["빈 목록"]
  nq --> nequal
  nequal --> nfound
  nequal --> nempty
```

strip()으로 양끝 공백을 지우고 전체 질문이 같은지 비교합니다. 참인 경로는 문서 객체의 본문과 출처를 함께 반환합니다.

## 관련 문서가 있는데도 검색이 실패하는 경우

Q03은 업로드 실패 시 행동을 묻습니다. p3에 분명한 절차가 있으므로 빈 검색을 올바른 답변 유보로 세면 안 됩니다. 첫 기준선의 알려진 실패로 남겨야 46장의 변경을 비교할 수 있습니다.


### 셀 02 · 검색 미적중 결과 진단

가상 안내의 업로드 실패 질문을 검색해 보세요.


In [2]:
failure_question = '과제 업로드가 실패했어요. 어떻게 해야 하나요?'
print(retrieve(failure_question))

[]


[]입니다. 검색 규칙이 Q01만 알아보기 때문에 p3까지 가지 못했습니다. 팀이 새로운 원문을 사용한다면 답이 있는 문장, 놓친 질문, 실제 반환 ID를 한 행으로 남기세요.


#### 같은 뜻으로 바꾼 질문이 실패하는 이유

<mark>질문 전체가 일치해야 하는 검색에서는 표현 하나가 달라도 빈 목록을 돌려줄 수 있습니다.</mark> 먼저 원래 질문으로 찾아 문단 ID를 기록한 뒤 같은 뜻의 새 표현을 보내세요. 자료가 존재하는데 표현 때문에 놓친 사례를 팀 기준선에 남길 수 있습니다.

선수 개념 · 19장 원래 질문과 검색 문단 연결하기

## 검색한 문단으로 답과 출처를 함께 만드는 기준선

앞 셀의 retrieve는 같은 뜻이어도 질문 문장이 달라지면 빈 목록을 반환합니다. 그 검색기를 그대로 두고 서비스가 실제로 돌려줄 답과 출처를 한 함수에 연결합니다.

### 셀 03 · 서비스의 첫 응답 만들기

정확히 일치하는 질문 한 건을 넣고, 응답 본문과 출처가 같은 레코드에서 나오는지 확인하세요.

In [3]:
def baseline_reply(question_text):
    hits = retrieve(question_text, k=1)
    if not hits:
        return {"answer": "제공된 자료에서 확인할 수 없습니다.", "citations": []}
    top = hits[0]
    return {"answer": top["text"], "citations": [(top["id"], top["page"])]}

first_reply = baseline_reply("과제는 어디에 내나요?")
print(first_reply["answer"])
print(first_reply["citations"])

과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다.
[('course-guide-v1:p2', 1)]


대출·일정 정보를 더해 답을 꾸미지 않고 찾은 p2의 문장을 그대로 내보냈습니다. 1은 쪽 번호이고 p2는 그 쪽 안의 문단입니다.

### 셀 04 · 말투를 바꿔 처음 깨지는 경계 기록하기

같은 사람이 “과제는 어디에 제출하죠?”라고 되묻습니다. 화면 문구보다 먼저 검색 후보를 확인한 뒤, 같은 질문에 대한 서비스 응답을 살펴보세요.

In [4]:
paraphrase = "과제는 어디에 제출하죠?"
print(retrieve(paraphrase))
print(baseline_reply(paraphrase))

[]
{'answer': '제공된 자료에서 확인할 수 없습니다.', 'citations': []}


빈 목록이 먼저 나옵니다. 화면은 정해진 안내를 올바르게 표시했지만 p2에는 답이 있습니다. 이 사례에서는 검색이 첫 실패 단계입니다.

### 팀의 Chroma 기준선으로 바꾸기

18장에서 만든 컬렉션이 있다면 retrieve의 내부에서 같은 질문을 조회해 문서 ID·본문·쪽수를 기존 반환 형태로 변환하세요. 첫 질문과 말투를 바꾼 질문을 다시 실행하고 p2가 각 검색 결과에 있는지 적으세요. 이 셀의 정확 일치 검색 결과와 팀의 벡터 검색 결과는 별도의 실행 기록으로 남깁니다.

<mark>답이 있는 질문에도 빈 검색 결과가 나올 수 있습니다.</mark> 같은 입력에 대해 후보 ID부터 남겨야 답변 문구를 고칠지 검색기를 고칠지 판단할 수 있습니다.

46장에는 기준 질문, 바꾼 질문, 기대 문단 p2, 처음 비어 버린 단계까지 전달하세요.

## 팀 사례에 적용하기 · 팀 질문을 기준선 검색기에 넣기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
def team_lookup(text, documents):
    if text.strip() == "과제는 어디에 내나요?" and "p2" in documents:
        return ["p2"]
    return []

runs = []
for row in incoming["cases"]:
    ids = team_lookup(row["question"], incoming["documents"])
    runs.append({**row, "hits": ids, "passed": ids == row["expected"]})
new = {"runs": runs, "documents": incoming["documents"]}
print([(row["id"], row["hits"], row["passed"]) for row in runs])

[('Q01', ['p2'], True), ('Q02', [], True), ('Q03', [], False)]


Q02와 Q03의 검색 결과가 모두 비더라도 정답 문단이 필요한 쪽은 Q03입니다. 기준선에 실패가 남아야 46장에서 같은 질문으로 개선 효과를 비교할 수 있습니다.

In [ ]:
output_file = project_root / "team-project/midterm/chapter45.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/midterm/chapter45.json


46장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

예시 인계 기록은 ‘Q01은 p2에서 답함, Q03은 p3가 있지만 찾지 못함’입니다. 실제 임베딩으로 만든 기준선에는 인덱스 판본과 검색 k도 함께 적습니다.
